# Lección 9.3 — Funciones de activación: sigmoide, tanh, ReLU y familia

> **Módulo 9 · Redes neuronales desde la intuición** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 9.2 (capas y profundidad), Lección 3.2 (función de pérdida), Lección 3.3 (descenso de gradiente)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar **para qué sirve** una función de activación y por qué tiene que ser **no lineal**.
- Reconocer la forma y el comportamiento de **sigmoide, tanh, ReLU, Leaky ReLU, GELU y SiLU**.
- Entender la **saturación** y las **ReLU muertas**, y por qué ReLU desplazó a la sigmoide en las capas ocultas.
- Elegir la **activación de salida** y la **pérdida** correctas según el problema (regresión, binaria, multiclase).

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits, make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(0)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo · dispositivo:", device)

---
## 1. 📖 La intuición: el "carácter" de cada neurona

Recuerda la neurona de la Lección 9.1: suma evidencias (`z`) y luego **decide**. La función de activación es **cómo reacciona** la neurona ante esa suma. Piensa en tres personas a las que les cuentas una noticia cada vez mejor:

| Persona | Cómo reacciona | Activación |
|---|---|---|
| 😐→😊 **La moderada** | Se entusiasma poco a poco, pero tiene un **tope**: por buena que sea la noticia, más allá de cierto punto ya no cambia | **Sigmoide** (entre 0 y 1) / **tanh** (entre −1 y 1) |
| 🤐→📣 **La directa** | Si la noticia es mala, **se calla** (0). Si es buena, la repite **tal cual**, sin tope | **ReLU** |
| 🔇 **La que se apagó** | Se calló una vez y ya nunca vuelve a hablar, digas lo que digas | Una **ReLU muerta** (Sección 4) |

En la Lección 9.2 viste lo más importante: **sin activación no lineal, apilar capas no sirve de nada** (todo colapsa en una sola capa lineal). La activación es la que "dobla" el espacio y permite fronteras curvas.

### ✋ Ejercicio 1 — Calentamiento
¿Cuál de estas funciones **no** sirve como activación de una capa oculta y por qué?
- a) `f(z) = max(0, z)`
- b) `f(z) = 3·z`
- c) `f(z) = 1 / (1 + e^(−z))`

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**b)** `3·z` es **lineal**: solo multiplica. Usarla entre capas es lo mismo que no usar nada; la red seguiría colapsando en una sola capa lineal (Lección 9.2). La a) es ReLU y la c) es la sigmoide: las dos "doblan" la recta en algún punto, y eso basta.
</details>

---
## 2. 📖 El concepto: la familia de activaciones

| Activación | Fórmula (intuición) | Rango de salida | Dónde se usa hoy |
|---|---|---|---|
| **Sigmoide** | `1 / (1 + e^(−z))`: "aplasta" todo entre 0 y 1 | (0, 1) | Salida de clasificación **binaria**; puertas de LSTM (Lección 11.4) |
| **Tanh** | Como la sigmoide pero centrada en 0 | (−1, 1) | Redes recurrentes (M11) |
| **ReLU** | `max(0, z)`: negativos → 0, positivos tal cual | [0, ∞) | **Capas ocultas** por defecto |
| **Leaky ReLU** | Como ReLU, pero los negativos se multiplican por 0.01 en lugar de anularse | (−∞, ∞) | Cuando hay ReLU muertas |
| **GELU** / **SiLU** (*Swish*) | Versiones "suaves" de ReLU, con una pequeña curva cerca de 0 | ≈ [−0.2, ∞) | **Transformers** y LLMs (M12–M13) |
| **Softmax** | Convierte un vector de puntuaciones en probabilidades que suman 1 | (0, 1), suma 1 | Salida **multiclase** |

In [ ]:
z = torch.linspace(-5, 5, 300)
activaciones = {
    "Sigmoide":   torch.sigmoid,
    "Tanh":       torch.tanh,
    "ReLU":       F.relu,
    "Leaky ReLU": lambda t: F.leaky_relu(t, 0.1),   # pendiente 0.1 (exagerada para que se vea)
    "GELU":       F.gelu,
    "SiLU":       F.silu,
}
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True)
for ax, (nombre, f) in zip(axes.ravel(), activaciones.items()):
    ax.plot(z, f(z), lw=2.5)
    ax.axhline(0, color="gray", lw=.8); ax.axvline(0, color="gray", lw=.8)
    ax.set_title(nombre); ax.grid(alpha=.3); ax.set_ylim(-1.5, 5)
plt.tight_layout(); plt.show()

### 🔮 Ejercicio 2 — Calcula sin calculadora
Completa la tabla **antes** de ejecutar la celda (aproximado basta):

| z | sigmoide | tanh | ReLU |
|---|---|---|---|
| −2 | ? | ? | ? |
| 0 | ? | ? | ? |
| 3 | ? | ? | ? |

✍️ *Mi tabla:*

In [ ]:
for valor in [-2.0, 0.0, 3.0]:
    t = torch.tensor(valor)
    print(f"z = {valor:>4}: sigmoide = {torch.sigmoid(t):.2f} · tanh = {torch.tanh(t):+.2f} · ReLU = {F.relu(t):.1f}")

<details><summary><b>👀 Ver respuesta</b></summary>

| z | sigmoide | tanh | ReLU |
|---|---|---|---|
| −2 | 0.12 | −0.96 | 0 |
| 0 | **0.50** | **0** | 0 |
| 3 | 0.95 | +1.00 | **3** |

Fíjate en `z = 3`: la sigmoide y la tanh ya están **casi en su techo** (0.95 y 0.995), mientras que la ReLU devuelve 3 sin más. Esa diferencia, "tener techo o no", es el tema de toda esta lección.
</details>

---
## 3. 📖 Cómo funciona por dentro: la pendiente es el "canal" del aprendizaje

Para aprender, la red necesita saber **cuánto cambia el error si mueve un peso un poquito** (el gradiente de la Lección 3.3). Esa señal viaja hacia atrás por la red (Lección 9.4) y, al cruzar cada neurona, **se multiplica por la pendiente de su activación**:

- Pendiente **grande** → la señal pasa con fuerza → la neurona aprende.
- Pendiente **casi 0** → la señal se apaga → la neurona **casi no aprende**.

PyTorch puede calcular la pendiente de cualquier activación por nosotros (con `autograd`, que verás en la Lección 9.4):

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, nombre in zip(axes, ["Sigmoide", "Tanh", "ReLU"]):
    zz = torch.linspace(-6, 6, 300, requires_grad=True)
    activaciones[nombre](zz).sum().backward()        # PyTorch calcula la pendiente en cada punto
    ax.plot(zz.detach(), activaciones[nombre](zz).detach(), lw=2, label="activación")
    ax.plot(zz.detach(), zz.grad, lw=2, ls="--", label="pendiente")
    ax.set_title(f"{nombre}: pendiente máxima = {zz.grad.max():.2f}"); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### 🔮 Ejercicio 3 — La pendiente de la sigmoide
En la gráfica, la pendiente máxima de la sigmoide es **0.25** (en `z = 0`).
1. ¿Cuánto vale su pendiente en `z = 10` (neurona muy "convencida")?
2. Si la señal cruza **10 capas** con sigmoide, multiplicándose cada vez por 0.25 (el **mejor** caso), ¿con qué fracción llega a la primera capa?

✍️ *Mis cuentas:*

In [ ]:
t = torch.tensor(10.0, requires_grad=True)
torch.sigmoid(t).backward()
print(f"Pendiente de la sigmoide en z = 10: {t.grad:.6f}")
for capas in [1, 5, 10, 20]:
    print(f"Tras {capas:>2} capas sigmoide (mejor caso): la señal se multiplica por 0.25^{capas} = {0.25 ** capas:.1e}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. En `z = 10` la pendiente es **≈ 0.000045**: la neurona está **saturada**. Por mucho que muevas sus pesos, su salida casi no cambia, así que casi no aprende.
2. Tras 10 capas: `0.25^10 ≈ 1e-6`, **una millonésima** en el mejor caso. Las primeras capas de una red profunda con sigmoide prácticamente **no reciben señal**: es el problema del **gradiente que desaparece** (*vanishing gradient*), que estudiarás a fondo en la Lección 9.7.

La ReLU tiene pendiente **exactamente 1** para todo `z > 0`: la señal cruza intacta. Por eso, desde ~2012, ReLU es la activación por defecto en capas ocultas.
</details>

### ▶️ Compruébalo: red profunda con sigmoide vs. ReLU
Entrenamos la **misma** red profunda (4 capas ocultas de 64 neuronas) sobre los dígitos escritos a mano de la Lección 9.2, cambiando **solo** la activación. Mismo optimizador (SGD, Lección 3.3), misma tasa de aprendizaje, mismos pasos.

In [ ]:
digitos = load_digits()
X_tr, X_te, y_tr, y_te = train_test_split(digitos.data, digitos.target, test_size=0.3,
                                          stratify=digitos.target, random_state=0)
esc = StandardScaler().fit(X_tr)
Xd_tr, Xd_te = (torch.tensor(esc.transform(a), dtype=torch.float32) for a in (X_tr, X_te))
yd_tr, yd_te = torch.tensor(y_tr), torch.tensor(y_te)

def crear_red(activacion, capas=4, ancho=64):
    piezas, n_ant = [], 64
    for _ in range(capas):
        piezas += [nn.Linear(n_ant, ancho), activacion()]
        n_ant = ancho
    piezas.append(nn.Linear(n_ant, 10))          # salida: 10 puntuaciones, SIN activación (ver Sección 5)
    return nn.Sequential(*piezas)

def entrenar(activacion, pasos=400, lr=0.1, capas=4, semilla=0):
    torch.manual_seed(semilla)
    red = crear_red(activacion, capas)
    opt = torch.optim.SGD(red.parameters(), lr=lr)
    historia = []
    for _ in range(pasos):
        perdida = nn.CrossEntropyLoss()(red(Xd_tr), yd_tr)
        opt.zero_grad(); perdida.backward(); opt.step()
        historia.append(perdida.item())
    with torch.no_grad():
        acc = (red(Xd_te).argmax(1) == yd_te).float().mean().item()
    return red, historia, acc

### 🔮 Ejercicio 4 — Predice antes de ejecutar
¿Cuál de las dos redes crees que aprenderá **más rápido**? ¿Alguna se quedará **atascada** al principio?

✍️ *Mi predicción:*

In [ ]:
for nombre, act in [("Sigmoide", nn.Sigmoid), ("ReLU", nn.ReLU)]:
    _, historia, acc = entrenar(act)
    plt.plot(historia, lw=2, label=f"{nombre} · exactitud en prueba {acc:.0%}")
plt.axhline(np.log(10), color="gray", ls=":", label="pérdida de adivinar al azar (≈ 2.30)")
plt.xlabel("paso"); plt.ylabel("pérdida"); plt.legend(); plt.grid(alpha=.3)
plt.title("Misma red, distinta activación"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

La red **ReLU** arranca algo despacio, pero hacia el paso 150–200 su pérdida se desploma y termina con ≈ 95 % en prueba. La red **sigmoide** se queda **plana** en ≈ 2.30 durante los 400 pasos: es la pérdida de **adivinar al azar** entre 10 clases (`ln 10 ≈ 2.30`). Su exactitud queda en el nivel del azar (≈ 10 %). La señal de aprendizaje se pierde al cruzar las 4 capas de sigmoide.

No es que la sigmoide "no pueda" aprender: con muchos más pasos, otra inicialización (Lección 9.7) o un optimizador adaptativo (Lección 9.5) acabaría arrancando. Pero ReLU lo hace **fácil**.
</details>

### ✏️ Ejercicio 5 — ¿Y con menos capas?
Cambia `CAPAS` a **1** y vuelve a ejecutar. ¿La sigmoide sigue atascada?

In [ ]:
CAPAS = 1     # ✏️ prueba 1, 2, 4, 6

for nombre, act in [("Sigmoide", nn.Sigmoid), ("Tanh", nn.Tanh), ("ReLU", nn.ReLU)]:
    _, historia, acc = entrenar(act, capas=CAPAS)
    print(f"{CAPAS} capa(s) ocultas · {nombre:<8} → pérdida final {historia[-1]:.3f} · exactitud en prueba {acc:.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

Con **1 capa** la sigmoide **sí aprende** (≈ 93 % en prueba): la señal solo cruza una sigmoide, así que no se apaga. Tanh y ReLU llegan a ≈ 97 % (y aprenden más rápido: su pérdida final es mucho menor). El problema de la sigmoide **crece con la profundidad**: con 4 o 6 capas vuelve a atascarse.

La tanh aguanta mejor que la sigmoide (su pendiente máxima es **1**, no 0.25, y está centrada en 0), pero también se satura en los extremos.
</details>

---
## 4. 📖 ¿Qué pasa si… una ReLU "se muere"?

ReLU tiene su propio problema. Si una neurona ReLU recibe `z < 0` para **todos** los ejemplos, su salida es siempre 0 y su pendiente también es 0. La señal de aprendizaje no le llega, así que sus pesos **nunca cambian**: queda muerta para siempre. Pasa, por ejemplo, tras un paso de aprendizaje demasiado grande que empuja su sesgo muy abajo.

La **Leaky ReLU** ("ReLU con fuga") intenta evitarlo: para `z < 0` devuelve `0.01·z` en vez de 0, así que siempre pasa un hilito de señal.

### 🔮 Ejercicio 6 — Predice antes de ejecutar
Simulamos el accidente: tomamos una red con una capa oculta de 32 neuronas y **forzamos sus sesgos a −10** (todas las neuronas "mudas" al empezar). La entrenamos con los dos círculos de la Lección 9.2 usando ReLU y Leaky ReLU.
1. ¿En cuál de las dos redes **cambiarán** los pesos de la capa oculta durante el entrenamiento?
2. ¿Alguna llegará a resolver los círculos?

✍️ *Mi predicción:*

In [ ]:
X_c, y_c = make_circles(n_samples=400, noise=0.1, factor=0.4, random_state=0)
Xc = torch.tensor(X_c, dtype=torch.float32)
yc = torch.tensor(y_c, dtype=torch.float32).unsqueeze(1)

SESGO_INICIAL = -10.0     # ✏️ prueba 0.0 (sin accidente)

for nombre, act in [("ReLU", nn.ReLU()), ("Leaky ReLU", nn.LeakyReLU(0.01))]:
    torch.manual_seed(0)
    red = nn.Sequential(nn.Linear(2, 32), act, nn.Linear(32, 1))
    with torch.no_grad():
        red[0].bias.fill_(SESGO_INICIAL)                      # 💥 el "accidente"
    pesos_antes = red[0].weight.detach().clone()
    opt = torch.optim.Adam(red.parameters(), lr=0.05)
    for _ in range(1000):
        perdida = nn.BCEWithLogitsLoss()(red(Xc), yc)
        opt.zero_grad(); perdida.backward(); opt.step()
    with torch.no_grad():
        cambio = (red[0].weight - pesos_antes).abs().max().item()   # ¿se movieron los pesos ocultos?
        acc = ((red(Xc) > 0).float() == yc).float().mean().item()
    print(f"{nombre:<10}: los pesos ocultos cambiaron como máximo {cambio:.2f} · exactitud {acc:.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. **ReLU:** los pesos ocultos cambiaron **exactamente 0.00** en 1,000 pasos. Ninguna neurona recibe señal, así que **ninguna se mueve**: la capa está congelada para siempre y entrenar más no sirve. **Leaky ReLU:** sus pesos sí se movieron (≈ 0.86): el hilito de señal llega.
2. **Ninguna** (≈ 50 %, azar). La Leaky ReLU recibe señal, pero como **todas** sus neuronas están en la zona `z < 0`, donde la Leaky es una simple recta, la red entera se comporta como una red **lineal** (Lección 9.2) y se queda atascada ahí. La fuga es un **seguro**, no una cura milagrosa.

✏️ Con `SESGO_INICIAL = 0.0` las dos llegan a ≈ 100 %. La lección práctica: unas pocas ReLU muertas en una red grande no son graves; si **muchas** mueren (lo verás como una pérdida que se estanca de golpe), el remedio es **evitar el accidente**: bajar la tasa de aprendizaje (Lección 9.6) y usar una buena inicialización (Lección 9.7), con Leaky ReLU o GELU como seguro extra.
</details>

---
## 5. 📖 La activación de **salida**: depende de la pregunta

Las capas **ocultas** usan casi siempre ReLU (o GELU). La capa de **salida** es distinta: su activación depende de **qué tipo de respuesta** quieres, y va de la mano con la **pérdida** (Lección 3.2):

| Problema | Ejemplo | Neuronas de salida | Activación de salida | Pérdida en PyTorch |
|---|---|---|---|---|
| **Regresión** | Precio de una casa | 1 | **Ninguna** (cualquier número) | `nn.MSELoss` (o `nn.L1Loss`) |
| **Binaria** | ¿Fraude? sí/no | 1 | **Sigmoide** → probabilidad | `nn.BCEWithLogitsLoss` (**ya incluye** la sigmoide) |
| **Multiclase** | ¿Qué dígito es? (0–9) | 10 (una por clase) | **Softmax** → probabilidades que suman 1 | `nn.CrossEntropyLoss` (**ya incluye** la softmax) |
| **Multietiqueta** | ¿Qué etiquetas tiene la foto? (playa, perro, sol…) | Una por etiqueta | **Sigmoide** en cada una | `nn.BCEWithLogitsLoss` |

> ⚠️ **La trampa más común de PyTorch:** `nn.CrossEntropyLoss` y `nn.BCEWithLogitsLoss` esperan las **puntuaciones crudas** de la última capa (llamadas **logits**) y aplican la softmax/sigmoide **por dentro** (de forma numéricamente más estable). Si tú añades `nn.Softmax()` al final de la red, la aplicas **dos veces** y la red aprende peor. La softmax/sigmoide solo se aplica **a mano** al predecir, cuando quieres ver probabilidades.

### 📖 La softmax, de cerca
La softmax toma las puntuaciones de cada clase, las vuelve positivas (`e^z`) y las divide por su suma, para que sumen 1. La clase con mayor puntuación se lleva la mayor probabilidad.

### 🔮 Ejercicio 7
Una red de 3 clases (🐱 gato, 🐶 perro, 🐰 conejo) da los logits `[2.0, 1.0, 0.1]`.
1. ¿Qué clase predice?
2. ¿La probabilidad del gato será **más** o **menos** del doble que la del perro?

✍️ *Mis predicciones:*

In [ ]:
logits = torch.tensor([2.0, 1.0, 0.1])          # ✏️ prueba [20.0, 10.0, 1.0] o [0.2, 0.1, 0.0]
probs = torch.softmax(logits, dim=0)
for clase, z_, p in zip(["gato", "perro", "conejo"], logits, probs):
    print(f"{clase:<7} logit {z_:>5.1f} → probabilidad {p:.2f}")
print(f"Suma de probabilidades: {probs.sum():.2f}")

# La pérdida que usa PyTorch es exactamente −log(probabilidad de la clase correcta):
clase_correcta = torch.tensor([0])               # supón que era un gato
print(f"\nCrossEntropyLoss(logits)       = {nn.CrossEntropyLoss()(logits.unsqueeze(0), clase_correcta):.4f}")
print(f"−log(prob. del gato) a mano    = {-torch.log(probs[0]):.4f}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Gato** (el logit más alto): probabilidades ≈ 0.66 / 0.24 / 0.10.
2. **Más del doble** (0.66 vs 0.24, casi el triple): la softmax usa exponenciales, así que **exagera** las diferencias. Con logits `[20, 10, 1]` el gato se lleva prácticamente el 100 %; con `[0.2, 0.1, 0.0]` las tres quedan cerca de 1/3.

Y la comprobación final muestra que `CrossEntropyLoss` aplicada a los **logits** da lo mismo (0.4170) que calcular la softmax y tomar `−log` de la probabilidad correcta: la softmax **ya está dentro** de la pérdida.
</details>

### ✋ Ejercicio 8 — Elige salida y pérdida
Para cada problema, di cuántas neuronas de salida, qué activación de salida y qué pérdida de PyTorch usarías:
1. Predecir la **temperatura** de mañana.
2. Decidir si un correo es **spam**.
3. Clasificar una prenda en **una** de 10 categorías.
4. Predecir los **minutos** de retraso de un vuelo (nunca negativos).

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **1 neurona, sin activación, `nn.MSELoss`** (la temperatura puede ser negativa).
2. **1 neurona, sigmoide (dentro de la pérdida), `nn.BCEWithLogitsLoss`.**
3. **10 neuronas, softmax (dentro de la pérdida), `nn.CrossEntropyLoss`.** Es justo el proyecto de este módulo (9.10).
4. **1 neurona, `nn.MSELoss` o `nn.L1Loss`.** Puedes dejarla sin activación y recortar los negativos al predecir, o usar una activación que solo dé positivos (como `nn.Softplus`). Ojo con poner ReLU a la salida: si la neurona cae en la zona negativa, se "muere" y la red deja de aprender (Sección 4).
</details>

---
## 6. 📖 Perillas: ¿qué activación elijo?

| Hiperparámetro | Qué controla | Opciones / efecto | Punto de partida |
|---|---|---|---|
| Activación oculta | Cómo "dobla" cada neurona la señal | Sigmoide/tanh: se saturan, malas en redes profundas · ReLU: rápida y simple, puede morir · Leaky ReLU: no muere · GELU/SiLU: suaves, algo más caras, estándar en Transformers | **ReLU** (GELU en Transformers) |
| `negative_slope` de Leaky ReLU | Cuánta señal deja pasar con `z < 0` | Más alta → menos riesgo de neuronas muertas pero menos "no lineal" · 0 → es una ReLU | 0.01 |
| Activación de salida | Tipo de respuesta | Ninguna / sigmoide / softmax según el problema (tabla de la Sección 5) | Según el problema |

### ✏️ Ejercicio 9 — Compara la familia completa
Entrenamos la red profunda de 4 capas (Sección 3) con **cuatro** activaciones. Antes de ejecutar, ordénalas de peor a mejor según tu intuición.

✍️ *Mi orden:*

In [ ]:
resultados = {}
for nombre, act in [("Sigmoide", nn.Sigmoid), ("Tanh", nn.Tanh), ("ReLU", nn.ReLU), ("GELU", nn.GELU)]:
    _, historia, acc = entrenar(act, pasos=400)        # ✏️ prueba pasos=1500
    resultados[nombre] = acc
    plt.plot(historia, lw=2, label=f"{nombre} · prueba {acc:.0%}")
plt.xlabel("paso"); plt.ylabel("pérdida"); plt.legend(); plt.grid(alpha=.3)
plt.title("Red de 4 capas ocultas: cuatro activaciones"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Peor → mejor: **sigmoide** (atascada en ≈ 10 %, azar) → **GELU** (arranca más despacio: ≈ 79 % a los 400 pasos) → **ReLU** y **tanh** (≈ 95 %). La tanh aprende bien porque su pendiente máxima es 1.

✏️ Con `pasos=1500`, tanh, ReLU y GELU terminan las tres en ≈ 96–97 %: las diferencias entre ellas son pequeñas y dependen del problema. La sigmoide, en cambio, **sigue en ≈ 10 %**: no es cuestión de paciencia. Lo que **no** quieres es una sigmoide en una red profunda.

✏️ Bonus: si pruebas `entrenar(nn.ReLU, capas=6)`, ¡incluso ReLU se queda casi atascada (≈ 30 %)! La profundidad trae sus propios problemas, que dependen también de cómo se **inicializan** los pesos: lo verás en la Lección 9.7.
</details>

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| La pérdida se queda plana en ≈ 2.30 (10 clases) o ≈ 0.69 (2 clases) | La red no aprende nada: p. ej. **sigmoides saturadas** en red profunda | Usar ReLU/GELU en las capas ocultas |
| La red aprende peor de lo esperado en multiclase | `nn.Softmax()` al final **y** `nn.CrossEntropyLoss` (softmax doble) | Quitar la softmax de la red; aplicarla solo al predecir |
| Regresión que nunca predice valores negativos | ReLU (o sigmoide) en la **capa de salida** | Salida sin activación |
| La pérdida se estanca de golpe y muchas neuronas dan siempre 0 | **ReLU muertas** (a menudo por tasa de aprendizaje demasiado alta) | Bajar la tasa (9.6), Leaky ReLU, revisar inicialización (9.7) |
| Error `Target ... is out of bounds` | Etiquetas que no van de 0 a `n_clases − 1`, o pocas neuronas de salida | Una neurona de salida por clase, etiquetas 0…K−1 |
| `BCEWithLogitsLoss` con valores de salida entre 0 y 1 | Aplicaste la sigmoide antes de la pérdida (sigmoide doble) | Pasar los logits crudos |

---
## 8. 📋 Ficha: las activaciones de un vistazo

| | Sigmoide | Tanh | ReLU | Leaky ReLU | GELU / SiLU | Softmax |
|---|---|---|---|---|---|---|
| **Rango** | (0, 1) | (−1, 1) | [0, ∞) | (−∞, ∞) | ≈ [−0.2, ∞) | (0, 1), suma 1 |
| **Pendiente máx.** | 0.25 | 1 | 1 | 1 | ≈ 1.1 | — |
| **✅ Fortaleza** | Da probabilidades | Centrada en 0 | Rápida, no se satura con `z > 0` | No muere | Suave, gran desempeño en Transformers | Probabilidades multiclase |
| **❌ Debilidad** | Se satura → gradientes que desaparecen | También se satura | Neuronas muertas | Un hiperparámetro más | Algo más cara de calcular | Solo para salida |
| **Úsala en…** | Salida binaria / multietiqueta | RNN (M11) | **Capas ocultas** (por defecto) | Si ves ReLU muertas | Transformers (M12) | Salida multiclase (dentro de la pérdida) |

## 9. 🧠 Resumen en 5 puntos
1. La activación es la **reacción** de cada neurona a su suma; tiene que ser **no lineal** para que apilar capas sirva.
2. La **pendiente** de la activación es el canal por el que llega la señal de aprendizaje: si es casi 0 (**saturación**), la neurona no aprende.
3. **Sigmoide y tanh se saturan**; en redes profundas la señal se apaga (**gradiente que desaparece**). Por eso **ReLU** es la opción por defecto en capas ocultas.
4. Las ReLU pueden **morir** (salida siempre 0); Leaky ReLU y GELU lo evitan.
5. La activación de **salida** depende del problema: ninguna (regresión), sigmoide (binaria), softmax (multiclase); en PyTorch, `BCEWithLogitsLoss` y `CrossEntropyLoss` **ya la incluyen**.

---
## 10. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 10.1 Repaso rápido (opción múltiple)

**1.** ¿Cuál es la pendiente de ReLU para `z = 5`?
- a) 0 · b) 0.25 · c) 1 · d) 5

**2.** Tu red de clasificación de 5 clases termina en `nn.Linear(64, 5)` y usas `nn.CrossEntropyLoss`. ¿Qué activación debes añadir **después** de la última capa para entrenar?
- a) `nn.Softmax()` · b) `nn.Sigmoid()` · c) `nn.ReLU()` · d) Ninguna

**3.** Una red de 10 capas con sigmoide se queda con la pérdida plana desde el inicio. ¿Cuál es la explicación más probable?
- a) Sobreajuste · b) Gradientes que desaparecen por saturación · c) Demasiados datos · d) La tasa de aprendizaje es 0

**4.** ¿Qué activación tiene **salidas negativas** para `z < 0` y aun así es casi lineal para `z > 0`?
- a) ReLU · b) Sigmoide · c) Leaky ReLU · d) Softmax

**5.** Para predecir el **precio** de una casa, la última capa debería ser…
- a) `nn.Linear(n, 1)` sin activación · b) `nn.Linear(n, 1)` + `nn.Sigmoid()` · c) `nn.Linear(n, 2)` + softmax · d) `nn.Linear(n, 1)` + `nn.Tanh()`

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **c)** Para todo `z > 0`, ReLU es `z`, cuya pendiente es 1.
2. **d)** `CrossEntropyLoss` aplica la softmax por dentro; añadirla sería aplicarla dos veces.
3. **b)** Clásico: cada sigmoide multiplica la señal por ≤ 0.25.
4. **c)** Para `z < 0` da `0.01·z` (negativo pequeño).
5. **a)** El precio puede ser cualquier número positivo grande; la sigmoide (b) lo limitaría a (0, 1) y la tanh (d) a (−1, 1).
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Si ReLU es tan buena, ¿por qué la sigmoide sigue apareciendo en la **salida** de los clasificadores binarios? ¿Podrías usar una ReLU ahí?

**B. Caso de análisis 💼** Un compañero entrenó una red para clasificar reseñas en 3 categorías (positiva, neutra, negativa). Te muestra su código: termina en `nn.Linear(128, 3), nn.Softmax(dim=1)` y usa `nn.CrossEntropyLoss`. Dice: *"Funciona: llega a 80 %, pero tarda mucho en entrenar y las probabilidades nunca pasan de 0.6"*. ¿Qué le dirías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** En la salida necesitas una **probabilidad** (entre 0 y 1) y la sigmoide la da de forma natural; además, combinada con la entropía cruzada, sus problemas de saturación se cancelan (por eso `BCEWithLogitsLoss` las calcula juntas). Una ReLU en la salida daría números sin techo que no son probabilidades y podría morir (siempre 0). La saturación es grave **dentro** de una red profunda, no en una única capa final.

**B.** Tiene una **softmax doble**: la suya y la que `CrossEntropyLoss` aplica por dentro. La segunda softmax recibe números entre 0 y 1 (muy parecidos entre sí), así que sus probabilidades quedan "aplastadas" hacia 1/3 y la señal de aprendizaje es débil: aprende lento. Solución: quitar `nn.Softmax` de la red y aplicar `torch.softmax(logits, dim=1)` solo al predecir. Verás este fallo "en vivo" en el recetario de la Lección 9.9.
</details>

### 10.3 🧩 Práctica integradora: "Arregla la red del becario"

Un becario construyó una red **profunda** para leer los dígitos de la oficina de correos (Lección 9.2), pero "no aprende". Vas a diagnosticarla y arreglarla en 5 pasos cortos. **No avances sin revisar cada respuesta.**

**Paso 1 ▶️ — La red del becario.** Ejecuta y mira la pérdida.

In [ ]:
torch.manual_seed(0)
red_becario = nn.Sequential(
    nn.Linear(64, 64), nn.Sigmoid(),
    nn.Linear(64, 64), nn.Sigmoid(),
    nn.Linear(64, 64), nn.Sigmoid(),
    nn.Linear(64, 64), nn.Sigmoid(),
    nn.Linear(64, 10), nn.Softmax(dim=1),
)

def entrenar_red(red, pasos=400, lr=0.1):
    opt = torch.optim.SGD(red.parameters(), lr=lr)
    for paso in range(pasos):
        perdida = nn.CrossEntropyLoss()(red(Xd_tr), yd_tr)
        opt.zero_grad(); perdida.backward(); opt.step()
        if paso % 100 == 0 or paso == pasos - 1:
            print(f"paso {paso:>3}: pérdida {perdida.item():.3f}")
    with torch.no_grad():
        print(f"Exactitud en prueba: {(red(Xd_te).argmax(1) == yd_te).float().mean():.0%}")

entrenar_red(red_becario)

✋ ¿Qué significa que la pérdida se quede en ≈ 2.30? Encuentra **dos** problemas en el código de la red.

✍️ *Mi diagnóstico:*

<details><summary><b>👀 Ver respuesta</b></summary>

La pérdida no se mueve de ≈ 2.30 = `ln(10)`: la red **adivina al azar** entre las 10 clases (exactitud ≈ 10 %). Dos problemas:
1. **Sigmoides en 4 capas ocultas** → saturación y gradiente que desaparece (Sección 3).
2. **`nn.Softmax` al final** junto con `nn.CrossEntropyLoss` → softmax doble (Sección 5).
</details>

**Paso 2 ✏️ — Arregla solo la salida.** Quita la `nn.Softmax` (borra la línea marcada) y entrena. ¿Basta?

In [ ]:
torch.manual_seed(0)
red_v2 = nn.Sequential(
    nn.Linear(64, 64), nn.Sigmoid(),
    nn.Linear(64, 64), nn.Sigmoid(),
    nn.Linear(64, 64), nn.Sigmoid(),
    nn.Linear(64, 64), nn.Sigmoid(),
    nn.Linear(64, 10),                 # ✏️ sin Softmax
)
entrenar_red(red_v2)

<details><summary><b>👀 Ver respuesta</b></summary>

**No basta**: la pérdida sigue en ≈ 2.30 y la exactitud en el nivel del azar. La softmax doble era un error real, pero el problema principal son las **sigmoides apiladas**. Un buen diagnóstico arregla **una cosa a la vez** para saber qué causó qué.
</details>

**Paso 3 ✏️ — Arregla las activaciones ocultas.** Cambia `ACT` por `nn.ReLU` y entrena.

In [ ]:
ACT = nn.ReLU        # ✏️ prueba nn.ReLU, nn.Tanh, nn.GELU

torch.manual_seed(0)
red_v3 = nn.Sequential(
    nn.Linear(64, 64), ACT(),
    nn.Linear(64, 64), ACT(),
    nn.Linear(64, 64), ACT(),
    nn.Linear(64, 64), ACT(),
    nn.Linear(64, 10),
)
entrenar_red(red_v3)

<details><summary><b>👀 Ver respuesta</b></summary>

Con ReLU la pérdida arranca lenta (≈ 2.14 en el paso 100), luego se desploma a ≈ 0.04 y la exactitud en prueba sube a ≈ **95 %**. Mismo número de capas, mismos pasos, misma tasa de aprendizaje: solo cambió la activación. Con `nn.Tanh` también funciona (≈ 95 %); con `nn.GELU` aprende más despacio y en estos 400 pasos se queda en ≈ 79 %.
</details>

**Paso 4 🔮 — ¿Y la softmax doble importaba?** Ahora vuelve a poner la `nn.Softmax` al final, pero **con ReLU** en las capas ocultas. ¿Qué crees que pasa con la pérdida final y la exactitud?

✍️ *Mi apuesta:*

In [ ]:
torch.manual_seed(0)
red_v4 = nn.Sequential(
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 10), nn.Softmax(dim=1),     # softmax doble otra vez
)
entrenar_red(red_v4)

<details><summary><b>👀 Ver respuesta</b></summary>

¡La red vuelve a quedarse **atascada** en ≈ 2.30 y la exactitud cae a ≈ 12 %! Con la softmax doble, la pérdida recibe números entre 0 y 1 muy parecidos entre sí y la señal de aprendizaje es tan débil que, con estos 400 pasos, la red apenas se mueve. Además, aunque acertara todo, la pérdida nunca podría bajar de ≈ 1.46. **El error es silencioso**: no salta ningún mensaje de error, solo aprende muchísimo peor (a veces solo "un poco peor", lo que lo hace aún más difícil de detectar). Es de los fallos más comunes en PyTorch (lo verás en el recetario de la Lección 9.9).
</details>

**Paso 5 ✋ — El informe.** Escríbele al becario **tres líneas** con las reglas que debe recordar.

✍️ *Mi informe:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. *En las capas ocultas usa **ReLU** (o GELU); evita la sigmoide salvo en redes de 1–2 capas.*
2. *La última capa de un clasificador devuelve **logits**, sin activación: `CrossEntropyLoss` ya aplica la softmax. Usa `torch.softmax` solo para mostrar probabilidades.*
3. *Si la pérdida se queda en `ln(número de clases)`, la red está adivinando al azar: revisa activaciones, salida y tasa de aprendizaje antes de entrenar más.*
</details>

---
## 📚 Para profundizar
- 🎮 **Interactivo:** *TensorFlow Playground*: cambia la activación (ReLU, Tanh, Sigmoid, Linear) en el mismo problema y compara la velocidad de aprendizaje.
- 📺 **Video:** StatQuest, "Neural Networks Pt. 3: ReLU In Action!" (inglés, con subtítulos).
- 📘 **Documentación:** PyTorch, sección *Non-linear Activations* de `torch.nn` (lista completa de activaciones) y las páginas de `CrossEntropyLoss` y `BCEWithLogitsLoss`.

---
⬅️ **Anterior:** [9.2 Capas y profundidad: por qué apilar funciona](9.2-capas-y-profundidad.ipynb) · ➡️ **Siguiente:** [9.4 Retropropagación (*backpropagation*): repartir la culpa del error](9.4-retropropagacion.ipynb)